# FalaPicto — Destilação + QLoRA + Quantização do SLM (Qwen3-0.6B)

**Status: pronto para executar.** (≈230 exemplos: 6 épocas ≈ 80 passos, ~5 min na T4) Rodar no Colab com GPU T4 (Runtime > Alterar tipo de ambiente de execução).

Pipeline da disciplina: dados do professor (Gemini) → curadoria → SFT com LoRA/QLoRA no Qwen3 → quantização (GGUF) → comparação FP16/INT8/INT4.

Nome técnico correto: *destilação a nível de sequência* (o estudante imita os textos do professor). A versão com logits/KL fica como extensão.

In [ ]:
!pip install -q unsloth trl datasets

In [ ]:
# Envie data/slm_train.jsonl (gerado por src/slm_dataset.py curar) para o Colab
from google.colab import files
files.upload()  # selecione slm_train_curto.jsonl (resposta rápida) ou slm_train.jsonl

In [ ]:
from unsloth import FastLanguageModel
import torch
MODEL='unsloth/Qwen3-0.6B'
model, tok = FastLanguageModel.from_pretrained(MODEL, max_seq_length=256, load_in_4bit=True)  # QLoRA: base em 4-bit NF4
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], random_state=42)

In [ ]:
import json, random
from datasets import Dataset
ROLE={'system':'system','human':'user','gpt':'assistant'}
import os
ARQ=next(f for f in ['slm_train_curto.jsonl','slm_train.jsonl'] if os.path.exists(f)); TAG='_curto' if 'curto' in ARQ else ''
print('usando', ARQ)
rows=[json.loads(l) for l in open(ARQ,encoding='utf-8')]
random.Random(0).shuffle(rows)
cut=int(len(rows)*0.9); tr, te = rows[:cut], rows[cut:]
def fmt(r):
    msgs=[{'role':ROLE[m['from']],'content':m['value']} for m in r['conversations']]
    return {'text': tok.apply_chat_template(msgs, tokenize=False)}
ds=Dataset.from_list([fmt(r) for r in tr])
print(len(tr),'treino |',len(te),'teste'); print(ds[0]['text'][:400])
print('MODO:', 'CURTO (só resposta)' if '<pensamento>' not in ds[0]['text'] else 'COM PENSAMENTO (lento)')


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer=SFTTrainer(model=model, tokenizer=tok, train_dataset=ds, args=SFTConfig(
    dataset_text_field='text', per_device_train_batch_size=8, gradient_accumulation_steps=2, num_train_epochs=6,
    learning_rate=2e-4, warmup_steps=5, logging_steps=5, optim='adamw_8bit', seed=42, output_dir='out', report_to='none'))
# loss só na resposta do assistente (loss masking, como na aula)
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
trainer.train()

In [ ]:
# Avaliação: exact match e fração de frases sem 'cards inventados' no conjunto de teste
import re
FastLanguageModel.for_inference(model)
def gerar(r):
    msgs=[{'role':ROLE[m['from']],'content':m['value']} for m in r['conversations'][:2]]
    ids=tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out=model.generate(ids, max_new_tokens=80, do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
def resp(t):
    m=re.search(r'<resposta>(.*?)</resposta>', t, re.S); return (m.group(1) if m else t).strip()
acertos=0
for r in te:
    gold=resp(r['conversations'][2]['value']); pred=resp(gerar(r)); acertos+= gold.lower()==pred.lower()
print('exact match:', acertos/len(te))

In [ ]:
# Quantização para o tablet: gere os 3 níveis e compare tamanho e qualidade
for q in ['f16','q8_0','q4_k_m']:
    model.save_pretrained_gguf(f'gguf_{q}', tok, quantization_method=q)
!ls -la gguf_*/*.gguf

In [ ]:
# Guarda os 3 .gguf no Google Drive (a sessão do Colab apaga tudo ao fechar)
import glob, os, shutil
from google.colab import drive
drive.mount('/content/drive')
destino = '/content/drive/MyDrive/falapicto_gguf' + TAG
os.makedirs(destino, exist_ok=True)
for f in glob.glob('/content/**/*.gguf', recursive=True):
    if '/drive/' in f: continue
    print(f, round(os.path.getsize(f)/1e6), 'MB'); shutil.copy(f, destino)
print(os.listdir(destino))


## Próximo passo
Rodar cada .gguf com `llama.cpp` (`llama-server -m modelo.gguf --port 8080`) e apontar `SLM_URL=http://localhost:8080/v1/chat/completions` na API. Medir latência e exact match de cada nível para a tabela do relatório.